# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [29]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.info()
spaceship.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [30]:
#spaceship.Destination.value_counts()
#drop ID and name columsn
spaceship = spaceship.drop(columns = ['PassengerId', 'Name'])

#clean column 'Cabin':
spaceship['Cabin'] = spaceship['Cabin'].apply(lambda x: str(x).strip().split("/")[0])

spaceship.head()

,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
0,Europa,False,B,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False
1,Earth,False,F,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True
2,Europa,False,A,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False
3,Europa,False,A,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False
4,Earth,False,F,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True


# Scaling

In [39]:
#Scaling
#Drop null values
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler, StandardScaler
spaceship.dropna(inplace = True)

#dummification

space_ship_cat = spaceship.select_dtypes('object')
space_ship_num = spaceship.select_dtypes('number')
# Dummisification = transformin nominal columns to num columns
#Create OneHotEncoder class instance
ohe = OneHotEncoder(sparse_output=False, drop = "first")

#find the unique values for every column
ohe.fit(space_ship_cat)

#output np array
space_ship_cat_np = ohe.transform(space_ship_cat)


#transform np array to data Frame
space_ship_cat_df = pd.DataFrame(space_ship_cat_np, columns = ohe.get_feature_names_out(), index = space_ship_cat.index)

#Scaling using stdandard scaler
#create instance
scaler = StandardScaler()

#fit the scaler
scaler.fit(spaceship_num)
#Transform scaler
space_ship_num_np = scaler.transform(spaceship_num)

#convert back the np array to df
spaceship_num_df = pd.DataFrame(space_ship_num_np, columns = space_ship_num.columns, index = space_ship_num.index)

spaceship_scaled = pd.concat([space_ship_cat_df, spaceship_num_df], axis = 1 )
spaceship_scaled.head()

,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Cabin_nan,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.696478,-0.335500,-0.287229,-0.283044,-0.274495,-0.269928
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.337043,-0.173745,-0.281849,-0.243585,0.199122,-0.230830
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,2.005605,-0.271688,1.850701,-0.283044,5.518474,-0.226387
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.283070,-0.335500,0.479819,0.302527,2.597403,-0.098429
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.888254,0.114149,-0.245379,-0.044712,0.212925,-0.268151


# Feature selection

In [59]:
target = spaceship['Transported']
#Check correlation between features and target col

correlation_feat_taget = spaceship_scaled.corrwith(target)
#correlation_feat_target
#drop columns with low correlation with target column
feat_col_to_keep = [row for row, value in correlation_feat_taget.items() if abs(value) > 0.1]

  
#correlation_feat_taget
spaceship_scaled_features = spaceship_scaled[feat_col_to_keep]
print("columns to keep:\n",feat_col_to_keep )
spaceship_scaled_features.head()

#Check correlation between features
correlation_feat_feat_spear =spaceship_scaled_features.corr(method = "spearman")
correlation_feat_feat_spearson =spaceship_scaled_features.corr(method = "pearson")

#Feature selection conclusion: There is not hogh correlation between features. We can keep the following col :  ['HomePlanet_Europa', 'CryoSleep_True', 'Cabin_B', 'Cabin_C', 'Destination_TRAPPIST-1e', 'RoomService', 'Spa', 'VRDeck']

columns to keep:
 ['HomePlanet_Europa', 'CryoSleep_True', 'Cabin_B', 'Cabin_C', 'Destination_TRAPPIST-1e', 'RoomService', 'Spa', 'VRDeck']


,HomePlanet_Europa,CryoSleep_True,Cabin_B,Cabin_C,Destination_TRAPPIST-1e,RoomService,Spa,VRDeck
HomePlanet_Europa,1.000000,0.094543,0.546486,0.526522,-0.153445,-0.207700,0.180897,0.208436
CryoSleep_True,0.094543,1.000000,0.120117,0.031276,-0.115780,-0.521932,-0.554834,-0.535512
Cabin_B,0.546486,0.120117,1.000000,-0.098310,-0.097235,-0.148126,0.021085,0.035808
Cabin_C,0.526522,0.031276,-0.098310,1.000000,-0.094651,-0.096505,0.119685,0.146870
Destination_TRAPPIST-1e,-0.153445,-0.115780,-0.097235,-0.094651,1.000000,0.099486,0.032617,0.004568
RoomService,-0.207700,-0.521932,-0.148126,-0.096505,0.099486,1.000000,0.247662,0.178451
Spa,0.180897,-0.554834,0.021085,0.119685,0.032617,0.247662,1.000000,0.438687
VRDeck,0.208436,-0.535512,0.035808,0.146870,0.004568,0.178451,0.438687,1.000000


**Perform Train Test Split**

In [66]:
#Perform Train Test Split
X_train, X_test, y_train, y_test = train_test_split(spaceship_scaled_features, target, test_size = 0.2, random_state = 0)


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [84]:
from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import BaggingClassifier
#Create predictor class
class_bag = BaggingClassifier(DecisionTreeClassifier(max_depth = 20),
                                   max_samples = 1000
                                  )
    
#train the model
class_bag.fit(X_train, y_train)

#Evaluate mode lperformance
y_pred_test = class_bag.predict(X_test)

print("Calssification Report:\n ", classification_report(y_test, y_pred_test))


Calssification Report:
                precision    recall  f1-score   support

       False       0.80      0.74      0.77       702
        True       0.75      0.81      0.78       683

    accuracy                           0.77      1385
   macro avg       0.77      0.77      0.77      1385
weighted avg       0.77      0.77      0.77      1385



- Random Forests

In [75]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

#create instance:
forest = RandomForestClassifier(max_depth = 20, n_estimators = 100)

#train model
forest.fit(X_train, y_train)

#Evaluate model
y_predict_test_forest= forest.predict(X_test)
print("Calssification Report:\n ", classification_report(y_test, y_predict_test_forest))


Calssification Report:
                precision    recall  f1-score   support

       False       0.81      0.71      0.76       702
        True       0.74      0.83      0.78       683

    accuracy                           0.77      1385
   macro avg       0.78      0.77      0.77      1385
weighted avg       0.78      0.77      0.77      1385



- Gradient Boosting

In [78]:
from sklearn.ensemble import GradientBoostingClassifier
#Init instance
gb_class = GradientBoostingClassifier(max_depth = 20, n_estimators=100)

#Train modem
gb_class.fit(X_train, y_train)

#evaluate
y_test_gb = gb_class.predict(X_test)
print("Calssification Report:\n ", classification_report(y_test, y_test_gb))


Calssification Report:
                precision    recall  f1-score   support

       False       0.81      0.65      0.72       702
        True       0.70      0.84      0.76       683

    accuracy                           0.74      1385
   macro avg       0.75      0.74      0.74      1385
weighted avg       0.75      0.74      0.74      1385



- Adaptive Boosting

In [83]:
from sklearn.ensemble import AdaBoostClassifier

#init instannce
adab = AdaBoostClassifier(DecisionTreeClassifier(max_depth = 20), n_estimators = 100)

#train model
adab.fit(X_train, y_train)

#evaluate model
y_adab_pred = adab.predict(X_test)
print ("Report Classification:\n", classification_report(y_test, y_adab_pred))

Report Classification:
               precision    recall  f1-score   support

       False       0.80      0.65      0.72       702
        True       0.70      0.83      0.76       683

    accuracy                           0.74      1385
   macro avg       0.75      0.74      0.74      1385
weighted avg       0.75      0.74      0.74      1385



Which model is the best and why?

In [ ]:
#The best model interms of balanced score is the bagging and pastin model. 